# 02 - Memory Decay Model Simulation

Design and validate the decay curve for the Pgramma memory engine.

**Core question**: Given an engram with initial `importance`, `access_count`, and `last_accessed`, what is its *effective weight* at any point in time? When should GC prune or compress it?

**Model basis**: Ebbinghaus forgetting curve adapted with rehearsal (access) reinforcement.

```
effective_weight = importance × decay_factor × access_boost

decay_factor = e^(-λ × days_since_last_access)
access_boost = 1 + α × ln(1 + access_count)
```

**GC decision boundaries** (from whitepaper):
- `effective_weight < 0.3` → physical deletion
- `effective_weight >= 0.3` but stale → LLM summarization & re-embedding

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dataclasses import dataclass

# --- Tunable hyperparameters ---
LAMBDA = 0.03      # decay rate (higher = faster forgetting)
ALPHA = 0.15       # access boost factor (higher = stronger rehearsal effect)
GC_PRUNE = 0.3     # below this effective weight → delete
GC_COMPRESS = 0.6  # above prune but below this & stale → summarize
STALE_DAYS = 30    # days without access to be considered "stale"

In [ ]:
def decay_factor(days: np.ndarray) -> np.ndarray:
    """Ebbinghaus-style exponential decay."""
    return np.exp(-LAMBDA * days)

def access_boost(access_count: int) -> float:
    """Logarithmic rehearsal reinforcement."""
    return 1.0 + ALPHA * np.log(1.0 + access_count)

def effective_weight(importance: float, days: np.ndarray, access_count: int = 0) -> np.ndarray:
    """Compute effective memory weight over time."""
    w = importance * decay_factor(days) * access_boost(access_count)
    return np.clip(w, 0.0, 1.0)

## 1. Decay Curves by Importance Tier

How fast do memories at each importance level decay without any access?

In [ ]:
days = np.linspace(0, 120, 500)

tiers = {
    "trivial (0.07)": 0.07,
    "factual (0.27)": 0.27,
    "personal (0.63)": 0.63,
    "core (0.93)": 0.93,
}

fig = go.Figure()
for label, imp in tiers.items():
    fig.add_trace(go.Scatter(
        x=days, y=effective_weight(imp, days, access_count=0),
        mode="lines", name=label, line=dict(width=2),
    ))

fig.add_hline(y=GC_PRUNE, line=dict(color="red", dash="dash", width=1),
              annotation_text=f"GC prune ({GC_PRUNE})", annotation_position="top left")
fig.add_hline(y=GC_COMPRESS, line=dict(color="orange", dash="dash", width=1),
              annotation_text=f"GC compress ({GC_COMPRESS})", annotation_position="top left")
fig.update_layout(
    title=f"Memory Decay Without Rehearsal (λ={LAMBDA})",
    xaxis_title="Days since last access", yaxis_title="Effective weight",
    yaxis_range=[0, 1.05], height=450, template="plotly_white",
)
fig.show()

# Print crossover points
for label, imp in tiers.items():
    w = effective_weight(imp, days, access_count=0)
    prune_idx = np.argmax(w < GC_PRUNE)
    if prune_idx > 0:
        print(f"  {label}: drops below prune threshold at day {days[prune_idx]:.0f}")
    else:
        print(f"  {label}: never drops below prune threshold in 120 days")

## 2. Rehearsal Effect (Access Boost)

How does repeated access slow down decay? Compare 0, 3, 10, 30 accesses on a mid-tier memory.

In [ ]:
base_importance = 0.50  # mid-tier personal memory
access_levels = [0, 3, 10, 30]

fig = go.Figure()
for ac in access_levels:
    w = effective_weight(base_importance, days, access_count=ac)
    boost = access_boost(ac)
    fig.add_trace(go.Scatter(
        x=days, y=w, mode="lines",
        name=f"access={ac} (boost={boost:.2f})", line=dict(width=2),
    ))

fig.add_hline(y=GC_PRUNE, line=dict(color="red", dash="dash", width=1),
              annotation_text=f"GC prune ({GC_PRUNE})", annotation_position="top left")
fig.add_hline(y=GC_COMPRESS, line=dict(color="orange", dash="dash", width=1),
              annotation_text=f"GC compress ({GC_COMPRESS})", annotation_position="top left")
fig.update_layout(
    title=f"Rehearsal Effect on Decay (importance={base_importance}, λ={LAMBDA}, α={ALPHA})",
    xaxis_title="Days since last access", yaxis_title="Effective weight",
    yaxis_range=[0, 1.05], height=450, template="plotly_white",
)
fig.show()

# Show boost values
print("Access boost values:")
for ac in [0, 1, 3, 5, 10, 20, 30, 50, 100]:
    print(f"  access_count={ac:>3d}  →  boost={access_boost(ac):.3f}")

## 3. GC Decision Heatmap

For every combination of `(importance, days_since_last_access)` with `access_count=0`, visualize the GC action: **keep**, **compress**, or **prune**.

In [ ]:
imp_range = np.linspace(0.0, 1.0, 200)
day_range = np.linspace(0, 120, 200)
imp_grid, day_grid = np.meshgrid(imp_range, day_range)

w_grid = imp_grid * np.exp(-LAMBDA * day_grid)  # access_count=0, no boost
w_grid = np.clip(w_grid, 0.0, 1.0)

# Decision map: 0=prune, 1=compress, 2=keep
decision = np.full_like(w_grid, 2)  # default keep
decision[w_grid < GC_COMPRESS] = 1   # compress zone
decision[w_grid < GC_PRUNE] = 0      # prune zone

# Only compress if also stale
decision[(w_grid >= GC_PRUNE) & (w_grid < GC_COMPRESS) & (day_grid < STALE_DAYS)] = 2

# Discrete colorscale: 0→red, 1→orange, 2→green
colorscale = [
    [0, "#ef5350"], [0.33, "#ef5350"],
    [0.33, "#ffa726"], [0.67, "#ffa726"],
    [0.67, "#66bb6a"], [1.0, "#66bb6a"],
]

fig = go.Figure()
fig.add_trace(go.Heatmap(
    z=decision, x=imp_range, y=day_range,
    colorscale=colorscale, zmin=0, zmax=2,
    colorbar=dict(tickvals=[0, 1, 2], ticktext=["Prune", "Compress", "Keep"]),
    hovertemplate="importance: %{x:.2f}<br>days: %{y:.0f}<br><extra></extra>",
))

# Contour lines for weight boundaries
fig.add_trace(go.Contour(
    z=w_grid, x=imp_range, y=day_range,
    contours=dict(start=GC_PRUNE, end=GC_COMPRESS, size=GC_COMPRESS - GC_PRUNE,
                  showlabels=True),
    line=dict(width=2, color="black", dash="dash"),
    showscale=False, contours_coloring="none",
))

fig.update_layout(
    title="GC Decision Map (access_count=0)",
    xaxis_title="Initial importance", yaxis_title="Days since last access",
    height=550, template="plotly_white",
)
fig.show()

## 4. Population Simulation

Simulate 500 engrams over 90 days. Each engram has random importance (weighted toward lower values) and gets randomly accessed. Track how the memory pool evolves and how GC would affect it.

In [ ]:
rng = np.random.default_rng(42)

@dataclass
class Engram:
    importance: float
    access_count: int = 0
    last_accessed_day: int = 0
    created_day: int = 0

    def effective_weight_at(self, current_day: int) -> float:
        days_elapsed = current_day - self.last_accessed_day
        w = self.importance * np.exp(-LAMBDA * days_elapsed) * access_boost(self.access_count)
        return min(w, 1.0)

    def gc_action(self, current_day: int) -> str:
        w = self.effective_weight_at(current_day)
        days_stale = current_day - self.last_accessed_day
        if w < GC_PRUNE:
            return "prune"
        if w < GC_COMPRESS and days_stale >= STALE_DAYS:
            return "compress"
        return "keep"

# Generate 500 engrams with realistic importance distribution
n_engrams = 500
importances = np.concatenate([
    rng.uniform(0.02, 0.19, size=int(n_engrams * 0.40)),  # 40% trivial
    rng.uniform(0.20, 0.49, size=int(n_engrams * 0.30)),  # 30% factual
    rng.uniform(0.50, 0.79, size=int(n_engrams * 0.20)),  # 20% personal
    rng.uniform(0.80, 0.98, size=int(n_engrams * 0.10)),  # 10% core
])
rng.shuffle(importances)

# Assign creation days (spread over first 30 days)
created_days = rng.integers(0, 30, size=len(importances))

engrams = [Engram(importance=float(imp), created_day=int(cd), last_accessed_day=int(cd))
           for imp, cd in zip(importances, created_days)]

# Simulate 90 days: each day, some engrams get accessed
sim_days = 90
daily_stats = {"day": [], "kept": [], "compress": [], "pruned": [], "avg_weight": []}

for day in range(sim_days):
    alive = [e for e in engrams if e.created_day <= day]
    if alive:
        probs = np.array([e.importance for e in alive])
        probs = probs / probs.sum()
        n_access = max(1, int(len(alive) * 0.05))
        accessed_indices = rng.choice(len(alive), size=n_access, replace=False, p=probs)
        for idx in accessed_indices:
            alive[idx].access_count += 1
            alive[idx].last_accessed_day = day

    actions = [e.gc_action(day) for e in alive]
    weights = [e.effective_weight_at(day) for e in alive]
    daily_stats["day"].append(day)
    daily_stats["kept"].append(actions.count("keep"))
    daily_stats["compress"].append(actions.count("compress"))
    daily_stats["pruned"].append(actions.count("prune"))
    daily_stats["avg_weight"].append(np.mean(weights) if weights else 0)

# --- Plot: Memory pool composition over time ---
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=["Memory Pool Composition Over Time (500 engrams, 90 days)",
                                    "Average Effective Weight of All Engrams"])

for name, color, data in [
    ("Keep", "#66bb6a", daily_stats["kept"]),
    ("Compress", "#ffa726", daily_stats["compress"]),
    ("Prune", "#ef5350", daily_stats["pruned"]),
]:
    fig.add_trace(go.Scatter(
        x=daily_stats["day"], y=data, name=name, mode="lines",
        line=dict(width=0.5, color=color), fillcolor=color,
        stackgroup="one",
    ), row=1, col=1)

fig.add_trace(go.Scatter(
    x=daily_stats["day"], y=daily_stats["avg_weight"],
    name="Avg weight", line=dict(color="#1565c0", width=2), showlegend=False,
), row=2, col=1)

fig.add_hline(y=GC_PRUNE, line=dict(color="red", dash="dash", width=1), row=2, col=1)
fig.add_hline(y=GC_COMPRESS, line=dict(color="orange", dash="dash", width=1), row=2, col=1)

fig.update_yaxes(title_text="Engram count", row=1, col=1)
fig.update_yaxes(title_text="Avg effective weight", row=2, col=1)
fig.update_xaxes(title_text="Day", row=2, col=1)
fig.update_layout(height=650, template="plotly_white")
fig.show()

# Final day stats
day_final = sim_days - 1
actions_final = [e.gc_action(day_final) for e in engrams]
print(f"\n--- Day {day_final} GC Summary ---")
print(f"  Keep:     {actions_final.count('keep'):>4d}")
print(f"  Compress: {actions_final.count('compress'):>4d}")
print(f"  Prune:    {actions_final.count('prune'):>4d}")

## 5. Hyperparameter Sensitivity

Sweep λ and α to understand how tuning affects GC pressure. Target: 40-60% of memories pruned by day 90.

In [ ]:
lambda_range = np.linspace(0.01, 0.15, 30)
alpha_range = np.linspace(0.05, 0.40, 30)
l_grid, a_grid = np.meshgrid(lambda_range, alpha_range)

# For each (λ, α), compute % of 500 engrams pruned at day 90
prune_pct = np.zeros_like(l_grid)

for i in range(len(alpha_range)):
    for j in range(len(lambda_range)):
        lam, alp = l_grid[i, j], a_grid[i, j]
        pruned = 0
        for e in engrams:
            days_elapsed = 89 - e.last_accessed_day
            boost = 1.0 + alp * np.log(1.0 + e.access_count)
            w = e.importance * np.exp(-lam * days_elapsed) * boost
            w = min(w, 1.0)
            if w < GC_PRUNE:
                pruned += 1
        prune_pct[i, j] = pruned / len(engrams) * 100

fig = go.Figure()
fig.add_trace(go.Heatmap(
    z=prune_pct, x=lambda_range, y=alpha_range,
    colorscale="RdYlGn_r", colorbar=dict(title="% Pruned"),
    hovertemplate="λ=%{x:.3f}<br>α=%{y:.3f}<br>pruned=%{z:.1f}%<extra></extra>",
))

# Contour lines at 40%, 50%, 60%
fig.add_trace(go.Contour(
    z=prune_pct, x=lambda_range, y=alpha_range,
    contours=dict(start=40, end=60, size=10, showlabels=True),
    line=dict(width=1.5, color="black"),
    showscale=False, contours_coloring="none", showlegend=False,
))

# Current config marker
fig.add_trace(go.Scatter(
    x=[LAMBDA], y=[ALPHA], mode="markers+text",
    marker=dict(size=8, color="white", line=dict(color="black", width=1.5)),
    text=[f"Current (λ={LAMBDA}, α={ALPHA})"], textposition="top right",
    showlegend=False,
))

fig.update_layout(
    title="% Engrams Pruned at Day 90 — Hyperparameter Sweep",
    xaxis_title="λ (decay rate)", yaxis_title="α (access boost factor)",
    height=600, template="plotly_white",
)
fig.show()

# Print current config stats
current_pruned = prune_pct[
    np.argmin(np.abs(alpha_range - ALPHA)),
    np.argmin(np.abs(lambda_range - LAMBDA)),
]
print(f"Current config (λ={LAMBDA}, α={ALPHA}): ~{current_pruned:.0f}% pruned at day 90")

## 6. Export: Rust Constants & Dummy Data

Export validated hyperparameters and generate a test SQLite database for Rust integration testing.

In [ ]:
import sqlite3
from datetime import datetime, timedelta, timezone

# --- Print Rust constants ---
print("// Auto-generated from 02_memory_decay.ipynb")
print(f"pub const DECAY_LAMBDA: f64 = {LAMBDA};")
print(f"pub const ACCESS_BOOST_ALPHA: f64 = {ALPHA};")
print(f"pub const GC_PRUNE_THRESHOLD: f64 = {GC_PRUNE};")
print(f"pub const GC_COMPRESS_THRESHOLD: f64 = {GC_COMPRESS};")
print(f"pub const GC_STALE_DAYS: u32 = {STALE_DAYS};")

# --- Generate dummy SQLite for Rust tests ---
db_path = "dummy_data/test_engrams.db"
conn = sqlite3.connect(db_path)

# Read and execute schema
with open("../src/db/schema.sql") as f:
    conn.executescript(f.read())

# Insert simulated engrams
now = datetime.now(timezone.utc)
emotions = ["neutral", "joy", "sadness", "trust", "disgust",
            "fear", "anger", "surprise", "anticipation", "contempt"]

for e in engrams:
    created = now - timedelta(days=89 - e.created_day)
    last_acc = now - timedelta(days=89 - e.last_accessed_day)
    emotion = rng.choice(emotions)

    conn.execute(
        "INSERT INTO engrams (content, emotion, importance, access_count, last_accessed, created_at) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        (
            f"Test engram imp={e.importance:.2f}",
            emotion,
            round(e.importance, 2),
            e.access_count,
            last_acc.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
            created.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        ),
    )

conn.commit()
print(f"\nDummy database written to: {db_path}")
print(f"  Engrams: {len(engrams)}")
print("  Schema: persona_config, episodic_memory, engrams")

conn.close()